# Notebook 05C — Focused GraphSAGE Optimization

## Dynamic Heterogeneous Graph Neural Network for Bank Marketing Prediction

### Purpose

Notebook 05B found the strongest candidate around:

- hidden dimension = 128
- dropout = 0.30
- learning rate = 0.0005
- validation PR-AUC ≈ 0.454855

This notebook performs a **small focused search around that candidate**.

The stale `NOTEBOOK_05_BASELINE` value of `0.111541` from Notebook 05B is deliberately ignored. The verified baseline discussed from Notebook 05 is approximately `0.448` PR-AUC.

### Strict rules

- Use the verified Notebook 04 graph.
- Train only using the training mask.
- Select checkpoints using validation PR-AUC.
- Never evaluate the test mask.
- Do not optimize the classification threshold.
- Do not freeze the final model.
- Preserve all experiments for comparison.

The output is a **candidate optimized model**, not a final production model.


# 1. Focused Experiment Design

Configurations tested:

| Experiment | Hidden | Dropout | Learning Rate |
|---|---:|---:|---:|
| 128_lr05_dropout20 | 128 | 0.20 | 0.0005 |
| 128_lr05_dropout30 | 128 | 0.30 | 0.0005 |
| 128_lr05_dropout40 | 128 | 0.40 | 0.0005 |
| 128_lr03_dropout30 | 128 | 0.30 | 0.0003 |
| 128_lr07_dropout30 | 128 | 0.30 | 0.0007 |
| 128_lr03_dropout40 | 128 | 0.40 | 0.0003 |

Primary metric: **validation PR-AUC**.

Secondary metrics: F1, precision, recall, ROC-AUC and validation loss.


In [ ]:
from pathlib import Path
import copy, json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

GRAPH_PATH = PROJECT_ROOT / "artifacts/graph/bank_heterodata.pt"
GRAPH_METADATA_PATH = PROJECT_ROOT / "artifacts/graph/graph_metadata.json"
BASELINE_CHECKPOINT_PATH = PROJECT_ROOT / "artifacts/models/graphsage_best_checkpoint.pt"

RESULTS_DIR = PROJECT_ROOT / "artifacts/results"
MODELS_DIR = PROJECT_ROOT / "artifacts/models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = RESULTS_DIR / "graphsage_focused_optimization_results.csv"
HISTORY_PATH = RESULTS_DIR / "graphsage_focused_optimization_histories.json"
PLOT_PATH = RESULTS_DIR / "graphsage_focused_pr_auc_comparison.png"
CHECKPOINT_PATH = MODELS_DIR / "graphsage_focused_candidate.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RANDOM_STATE = 42
MAX_EPOCHS = 150
PATIENCE = 20
PRIMARY_METRIC = "val_pr_auc"

# Verified approximate Notebook 05 result discussed during review.
# Do NOT use the stale 0.111541 value from 05B metadata.
VERIFIED_BASELINE_PR_AUC = 0.448

for p in [GRAPH_PATH, GRAPH_METADATA_PATH, BASELINE_CHECKPOINT_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Required input missing: {p}")

print("Device:", DEVICE)
print("Graph:", GRAPH_PATH)


In [ ]:
# Load verified graph
try:
    data = torch.load(GRAPH_PATH, map_location="cpu", weights_only=False)
except TypeError:
    data = torch.load(GRAPH_PATH, map_location="cpu")

if not isinstance(data, HeteroData):
    raise TypeError(f"Expected HeteroData, got {type(data)}")

with open(GRAPH_METADATA_PATH, "r", encoding="utf-8") as f:
    graph_metadata = json.load(f)

expected_nodes = {"customer","job","education","marital","contact","month"}
assert set(data.node_types) == expected_nodes

data = data.to(DEVICE)

train_mask = data["customer"].train_mask.bool()
val_mask = data["customer"].val_mask.bool()
test_mask = data["customer"].test_mask.bool()

assert torch.all(
    train_mask.to(torch.int8)
    + val_mask.to(torch.int8)
    + test_mask.to(torch.int8) == 1
)

# Explicit test protection: this notebook never evaluates test_mask.
test_evaluation_count = 0

print(data)
print("Test evaluations:", test_evaluation_count)


# 2. Model Definition

The architecture remains the same as Notebook 05/05B:

`HeteroConv → ReLU → Dropout → HeteroConv → ReLU → Dropout → Customer classifier`

Only the focused hyperparameters change.


In [ ]:
class HeteroGraphSAGE(nn.Module):
    def __init__(self, metadata, hidden_dim=128, dropout=0.30):
        super().__init__()
        _, edge_types = metadata
        self.dropout = dropout

        self.conv1 = HeteroConv({
            edge_type: SAGEConv((-1, -1), hidden_dim, aggr="mean")
            for edge_type in edge_types
        }, aggr="sum")

        self.conv2 = HeteroConv({
            edge_type: SAGEConv((-1, -1), hidden_dim, aggr="mean")
            for edge_type in edge_types
        }, aggr="sum")

        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.conv1(x_dict, edge_index_dict)
        x_dict = {
            k: F.dropout(F.relu(v), p=self.dropout, training=self.training)
            for k, v in x_dict.items()
        }

        x_dict = self.conv2(x_dict, edge_index_dict)
        x_dict = {
            k: F.dropout(F.relu(v), p=self.dropout, training=self.training)
            for k, v in x_dict.items()
        }

        return self.classifier(x_dict["customer"]).squeeze(-1)


In [ ]:
def metrics(y_true, probabilities, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    probabilities = np.asarray(probabilities)
    predictions = (probabilities >= threshold).astype(int)

    return {
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(y_true, probabilities),
    }

EXPERIMENTS = [
    {"experiment":"128_lr05_dropout20","hidden_dim":128,"dropout":0.20,"learning_rate":0.0005,"weight_decay":1e-4},
    {"experiment":"128_lr05_dropout30","hidden_dim":128,"dropout":0.30,"learning_rate":0.0005,"weight_decay":1e-4},
    {"experiment":"128_lr05_dropout40","hidden_dim":128,"dropout":0.40,"learning_rate":0.0005,"weight_decay":1e-4},
    {"experiment":"128_lr03_dropout30","hidden_dim":128,"dropout":0.30,"learning_rate":0.0003,"weight_decay":1e-4},
    {"experiment":"128_lr07_dropout30","hidden_dim":128,"dropout":0.30,"learning_rate":0.0007,"weight_decay":1e-4},
    {"experiment":"128_lr03_dropout40","hidden_dim":128,"dropout":0.40,"learning_rate":0.0003,"weight_decay":1e-4},
]

display(pd.DataFrame(EXPERIMENTS))


# 3. Train One Focused Experiment

Each experiment starts from fresh weights.

Class weighting is calculated only from the training mask.

Early stopping is based only on validation PR-AUC.


In [ ]:
def train_experiment(config):
    random.seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    torch.manual_seed(RANDOM_STATE)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_STATE)

    y = data["customer"].y.long()
    train_y = y[train_mask]

    neg = int((train_y == 0).sum())
    pos = int((train_y == 1).sum())
    pos_weight = torch.tensor([neg / pos], dtype=torch.float32, device=DEVICE)

    model = HeteroGraphSAGE(
        data.metadata(),
        hidden_dim=config["hidden_dim"],
        dropout=config["dropout"]
    ).to(DEVICE)

    model.train()
    with torch.no_grad():
        model(data.x_dict, data.edge_index_dict)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"]
    )

    best_pr = -np.inf
    best_epoch = -1
    best_state = None
    stale = 0
    history = []
    start = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        logits = model(data.x_dict, data.edge_index_dict)
        train_loss = criterion(
            logits[train_mask],
            y[train_mask].float()
        )
        train_loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(data.x_dict, data.edge_index_dict)

        val_loss = criterion(
            val_logits[val_mask],
            y[val_mask].float()
        ).item()

        probs = torch.sigmoid(val_logits[val_mask]).detach().cpu().numpy()
        targets = y[val_mask].detach().cpu().numpy()
        m = metrics(targets, probs)

        history.append({
            "epoch": epoch,
            "train_loss": float(train_loss.item()),
            "val_loss": float(val_loss),
            "val_accuracy": float(m["accuracy"]),
            "val_precision": float(m["precision"]),
            "val_recall": float(m["recall"]),
            "val_f1": float(m["f1"]),
            "val_roc_auc": float(m["roc_auc"]),
            "val_pr_auc": float(m["pr_auc"]),
        })

        if m["pr_auc"] > best_pr:
            best_pr = m["pr_auc"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1

        if stale >= PATIENCE:
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        logits = model(data.x_dict, data.edge_index_dict)

    val_loss = criterion(
        logits[val_mask],
        y[val_mask].float()
    ).item()

    probs = torch.sigmoid(logits[val_mask]).detach().cpu().numpy()
    targets = y[val_mask].detach().cpu().numpy()
    m = metrics(targets, probs)

    return {
        **config,
        "best_epoch": int(best_epoch),
        "best_val_loss": float(val_loss),
        "val_accuracy": float(m["accuracy"]),
        "val_precision": float(m["precision"]),
        "val_recall": float(m["recall"]),
        "val_f1": float(m["f1"]),
        "val_roc_auc": float(m["roc_auc"]),
        "val_pr_auc": float(m["pr_auc"]),
        "training_seconds": float(time.time() - start),
        "best_state_dict": best_state,
        "history": history,
    }


# 4. Run Focused Experiments

All experiments use the same graph and the same train/validation split.


In [ ]:
focused_results = []

for i, config in enumerate(EXPERIMENTS, 1):
    print("=" * 80)
    print(f"Experiment {i}/{len(EXPERIMENTS)}: {config['experiment']}")
    print(
        f"hidden={config['hidden_dim']} | "
        f"dropout={config['dropout']} | "
        f"lr={config['learning_rate']}"
    )

    result = train_experiment(config)
    focused_results.append(result)

    print(
        f"Best epoch={result['best_epoch']} | "
        f"PR-AUC={result['val_pr_auc']:.6f} | "
        f"ROC-AUC={result['val_roc_auc']:.6f} | "
        f"F1={result['val_f1']:.6f}"
    )


In [ ]:
results_df = pd.DataFrame([
    {
        k: v for k, v in r.items()
        if k not in {"best_state_dict", "history"}
    }
    for r in focused_results
]).sort_values("val_pr_auc", ascending=False).reset_index(drop=True)

display(results_df[
    [
        "experiment","hidden_dim","dropout","learning_rate",
        "best_epoch","val_f1","val_roc_auc","val_pr_auc",
        "val_precision","val_recall","best_val_loss"
    ]
])


# 5. Compare Against the Correct Baseline

The old 05B row showing PR-AUC `0.111541` is not used.

The verified baseline discussed during review is approximately:

`PR-AUC = 0.448`

If the exact original Notebook 05 value is available, replace the constant in the configuration cell with that exact value.


In [ ]:
best = max(focused_results, key=lambda r: r["val_pr_auc"])

improvement = best["val_pr_auc"] - VERIFIED_BASELINE_PR_AUC
relative_pct = improvement / VERIFIED_BASELINE_PR_AUC * 100

comparison_df = pd.DataFrame([
    {
        "model": "Notebook 05 verified baseline",
        "val_pr_auc": VERIFIED_BASELINE_PR_AUC,
        "val_f1": np.nan,
        "val_roc_auc": np.nan,
    },
    {
        "model": best["experiment"],
        "val_pr_auc": best["val_pr_auc"],
        "val_f1": best["val_f1"],
        "val_roc_auc": best["val_roc_auc"],
    }
])

display(comparison_df)

print(f"Baseline PR-AUC : {VERIFIED_BASELINE_PR_AUC:.6f}")
print(f"Best PR-AUC     : {best['val_pr_auc']:.6f}")
print(f"Improvement     : {improvement:+.6f}")
print(f"Relative change : {relative_pct:+.2f}%")


In [ ]:
# Save complete experiment results
results_df.to_csv(RESULTS_PATH, index=False)

with open(HISTORY_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {r["experiment"]: r["history"] for r in focused_results},
        f,
        indent=2
    )

# Plot validation PR-AUC
plot_df = results_df.sort_values("val_pr_auc", ascending=True)

plt.figure(figsize=(11, 6))
plt.barh(plot_df["experiment"], plot_df["val_pr_auc"])
plt.axvline(
    VERIFIED_BASELINE_PR_AUC,
    linestyle="--",
    label="Notebook 05 verified baseline"
)
plt.xlabel("Validation PR-AUC")
plt.ylabel("Experiment")
plt.title("Focused GraphSAGE Optimization")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_PATH, dpi=150, bbox_inches="tight")
plt.show()

print("Saved:", RESULTS_PATH)
print("Saved:", HISTORY_PATH)
print("Saved:", PLOT_PATH)


# 6. Save Candidate Only If It Beats the Baseline

The winning checkpoint is saved as a **candidate**.

It is not the final model and must not be deployed yet.


In [ ]:
if best["val_pr_auc"] > VERIFIED_BASELINE_PR_AUC:
    candidate = {
        "model_state_dict": best["best_state_dict"],
        "model_config": {
            "model_class": "HeteroGraphSAGE",
            "hidden_dim": best["hidden_dim"],
            "dropout": best["dropout"],
            "learning_rate": best["learning_rate"],
            "weight_decay": best["weight_decay"],
            "max_epochs": MAX_EPOCHS,
            "patience": PATIENCE,
            "primary_metric": PRIMARY_METRIC,
            "threshold_for_training_metrics": 0.5,
            "node_types": list(data.node_types),
            "edge_types": [list(e) for e in data.edge_types],
            "customer_feature_dimension": int(data["customer"].x.shape[1]),
        },
        "best_epoch": int(best["best_epoch"]),
        "validation_metrics": {
            "accuracy": float(best["val_accuracy"]),
            "precision": float(best["val_precision"]),
            "recall": float(best["val_recall"]),
            "f1": float(best["val_f1"]),
            "roc_auc": float(best["val_roc_auc"]),
            "pr_auc": float(best["val_pr_auc"]),
            "loss": float(best["best_val_loss"]),
        },
        "baseline_pr_auc": VERIFIED_BASELINE_PR_AUC,
        "graph_metadata": graph_metadata,
    }

    torch.save(candidate, CHECKPOINT_PATH)
    print("Candidate saved:", CHECKPOINT_PATH)
else:
    print("Focused optimization did not beat the verified baseline.")


# 7. Final Verification and Stop Point

Notebook 05C is complete when:

- focused experiments finished
- validation PR-AUC was used for selection
- the stale `0.111541` baseline was ignored
- results were saved
- test evaluation count remains zero
- any saved checkpoint is labeled a candidate

## STOP

Do not evaluate the test set yet.

Do not optimize the classification threshold yet.

Do not call the candidate final.

After reviewing the results, the next decision is either another controlled experiment or:

`MODEL IS SATISFACTORY`

which starts the final model-freeze stage.


In [ ]:
assert len(focused_results) == len(EXPERIMENTS)
assert RESULTS_PATH.exists()
assert HISTORY_PATH.exists()
assert PLOT_PATH.exists()
assert test_evaluation_count == 0

assert all(0 <= r["val_pr_auc"] <= 1 for r in focused_results)
assert all(0 <= r["val_roc_auc"] <= 1 for r in focused_results)

if best["val_pr_auc"] > VERIFIED_BASELINE_PR_AUC:
    assert CHECKPOINT_PATH.exists()

print("=" * 85)
print("NOTEBOOK 05C VERIFICATION PASSED")
print("=" * 85)
print("Experiments:", len(focused_results))
print("Baseline PR-AUC:", f"{VERIFIED_BASELINE_PR_AUC:.6f}")
print("Best experiment:", best["experiment"])
print("Best PR-AUC:", f"{best['val_pr_auc']:.6f}")
print("Best ROC-AUC:", f"{best['val_roc_auc']:.6f}")
print("Best F1:", f"{best['val_f1']:.6f}")
print("Improvement:", f"{improvement:+.6f}")
print("Test evaluations:", test_evaluation_count)
print("STOP: review results before final freeze.")
